# Descriptive Statistics & Predictor Readiness Check
## Postoperative Complications After Esophageal Cancer Surgery -- DUCA Registry 2016-2025

---

This notebook performs a full descriptive analysis of the predictor variables selected for predicting postoperative complications after esophageal cancer surgery.

**Goals of this notebook:**
1. Load and merge the two dataset sheets
2. Describe the distribution of each predictor (mean, median, missing values, outliers)
3. Visualize distributions for continuous and categorical variables
4. Check each variable for issues (missingness, outliers, data type problems)
5. Produce a final **readiness summary table** -- which variables are ready for modeling and which need attention

**Predictor groups covered:**
- Demographics
- Fitness & Lung Function
- Nutrition & Body Composition
- Comorbidities & Lifestyle
- Prior Surgical History
- Tumor & Disease Characteristics
- Neoadjuvant Treatment
- Surgical Factors


---
## 1. Setup & Data Loading

First we import the required libraries and load both dataset sheets from the Excel file.

Before merging, we standardise the **`upn`** (Uniek Pati nt Nummer) column:
- Convert to string
- Pad with leading zeros to ensure all IDs are exactly **7 digits** (e.g. `12345`   `0012345`)

This prevents mismatches during the merge caused by IDs stored as integers in one sheet and strings in the other.

The two sheets are then merged on `upn` using a **left join** -- all 1,181 patients in the merged cohort are kept, and the extra variables from sheet 2 are added where available.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Blues_d')

# Source data is not included in this repository (governed by DUCA/AUMC data agreement) -- point this at your own extract.
FILE_PATH = "../data/duca_esophagectomy_raw.xlsx"

In [ ]:
# Load both sheets
df1 = pd.read_excel(FILE_PATH, sheet_name=0)
df2 = pd.read_excel(FILE_PATH, sheet_name=1)

print(f"Sheet 1: {df1.shape[0]} rows, {df1.shape[1]} columns")
print(f"Sheet 2: {df2.shape[0]} rows, {df2.shape[1]} columns")


### 1.1 Standardise Patient IDs

We pad the `upn` column in both sheets to 7 digits with leading zeros.
This ensures IDs like `12345` and `0012345` are treated as the same patient during the merge.


In [ ]:
# Standardise upn to 7-digit string in both sheets
def pad_upn(df, col='upn', digits=7):
    df = df.copy()
    df[col] = df[col].astype(str).str.strip()          # convert to string, remove whitespace
    df[col] = df[col].str.split('.').str[0]             # remove decimal if read as float (e.g. '12345.0' -> '12345')
    df[col] = df[col].str.zfill(digits)                 # pad with leading zeros to 7 digits
    return df

df1 = pad_upn(df1)
df2 = pad_upn(df2)

# Quick check
print("Sample upn values after padding:")
print("  Sheet 1:", df1['upn'].head(5).tolist())
print("  Sheet 2:", df2['upn'].head(5).tolist())

# Check IDs shorter/longer than 7 digits (should be 0 after padding)
weird_1 = (df1['upn'].str.len() != 7).sum()
weird_2 = (df2['upn'].str.len() != 7).sum()
print(f"\nIDs not exactly 7 digits -- Sheet 1: {weird_1}, Sheet 2: {weird_2}")

print(f"\nPatients matchable between sheets: {df1['upn'].isin(df2['upn']).sum()}")


### 1.2 Merge the Two Sheets


In [ ]:
# Merge on upn (left join -- keep all sheet 1 patients)
df = df1.merge(df2, on='upn', how='left')
print(f"Merged dataset: {df.shape[0]} rows, {df.shape[1]} columns")

x_cols = [c for c in df.columns if c.endswith('_x')]
y_cols = [c for c in df.columns if c.endswith('_y')]

df = df.drop(columns=y_cols)
df = df.rename(columns={c: c[:-2] for c in x_cols})

print(f"Patients with data from both sheets: {df.dropna(subset=df2.columns.drop('upn').tolist(), how='all').shape[0]}")
print(f"Patients with only sheet 1 data: {df[df2.columns.drop('upn')[0]].isna().sum() if df2.columns.drop('upn')[0] in df.columns else 'N/A'}")


In [ ]:
# Compute derived variables

# --- Age at surgery ---
if 'datok' in df.columns and 'gebdat' in df.columns:
    try:
        # Convert to datetime -- store in new columns to avoid overwriting originals
        datok_dt  = pd.to_datetime(df['datok'],  errors='coerce', dayfirst=True)
        gebdat_dt = pd.to_datetime(df['gebdat'], errors='coerce', dayfirst=True)

        # Fallback: Excel stores dates as serial integers (days since 1899-12-30)
        if datok_dt.isna().mean() > 0.5:
            datok_dt  = pd.to_datetime(df['datok'],  unit='D', origin='1899-12-30', errors='coerce')
        if gebdat_dt.isna().mean() > 0.5:
            gebdat_dt = pd.to_datetime(df['gebdat'], unit='D', origin='1899-12-30', errors='coerce')

        df['age_at_op'] = (datok_dt - gebdat_dt).dt.days // 365

        valid = df['age_at_op'].notna().sum()
        print(f"age_at_op: {valid} valid values")
        print(f"  Range: {df['age_at_op'].min():.0f} - {df['age_at_op'].max():.0f} years")
        print(f"  Mean:  {df['age_at_op'].mean():.1f} years")

        # Sanity check -- flag implausible ages
        bad = ((df['age_at_op'] < 18) | (df['age_at_op'] > 100)).sum()
        if bad > 0:
            print(f"    {bad} implausible ages (<18 or >100) -- check source data")

    except Exception as e:
        print(f"Could not compute age_at_op: {e}")
        df['age_at_op'] = np.nan
else:
    print("Warning: datok or gebdat not found -- age_at_op set to NaN")
    df['age_at_op'] = np.nan

# --- BMI ---
if 'BMI' not in df.columns and 'gewicht' in df.columns and 'lengte' in df.columns:
    gewicht = pd.to_numeric(df['gewicht'], errors='coerce')
    lengte  = pd.to_numeric(df['lengte'],  errors='coerce')
    df['BMI'] = gewicht / ((lengte / 100) ** 2)
    print(f"\nBMI: {df['BMI'].notna().sum()} valid values")
    print(f"  Range: {df['BMI'].min():.1f} - {df['BMI'].max():.1f} kg/m ")

print(f"\nFinal dataset ready: {df.shape[0]} patients, {df.shape[1]} columns")


---
## 2. Target Variables Overview

Before describing the predictors, we briefly check the distribution of our outcome variables.
These are **not predictors** -- they are what we want to predict.


In [ ]:
# Complication outcomes
comp_cols = ['compl', 'comppulm', 'compcar', 'compgas', 'compuro',
             'compthrom', 'compneu', 'compinfect', 'compwond', 'compchylek', 'compand']

comp_labels = {
    'compl': 'Any complication',
    'comppulm': 'Pulmonary',
    'compcar': 'Cardiac',
    'compgas': 'Gastrointestinal',
    'compuro': 'Urological',
    'compthrom': 'Thrombosis',
    'compneu': 'Neurological',
    'compinfect': 'Infection',
    'compwond': 'Wound',
    'compchylek': 'Chyle leak',
    'compand': 'Anastomotic leak'
}

comp_present = [c for c in comp_cols if c in df.columns]

print("=" * 55)
print(f"{'Outcome':<20} {'N events':>10} {'% of total':>12}")
print("=" * 55)
for col in comp_present:
    n = df[col].sum()
    pct = df[col].mean() * 100
    label = comp_labels.get(col, col)
    flag = "   <100 events" if n < 100 else ""
    print(f"{label:<20} {int(n):>10} {pct:>11.1f}%{flag}")
print("=" * 55)
print(f"\nNote: Outcomes with <100 events may be underpowered for")
print(f"standalone models with the full predictor set.")


In [ ]:
# Clavien-Dindo severity distribution
clav_cols = [c for c in df.columns if c.startswith('clav')]
if clav_cols:
    df['derived_clavien'] = df[clav_cols].apply(pd.to_numeric, errors='coerce').replace({9: np.nan}).max(axis=1)
    df['derived_clavien_bin'] = np.where(df['derived_clavien'].between(1, 2), 0,
                                 np.where(df['derived_clavien'].between(3, 7), 1, np.nan))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grade distribution
if 'derived_clavien' in df.columns:
    grade_counts = df['derived_clavien'].value_counts(dropna=False).sort_index()
    axes[0].bar([str(x) for x in grade_counts.index], grade_counts.values, color='steelblue', edgecolor='white')
    axes[0].set_title('Clavien-Dindo Grade Distribution', fontweight='bold', fontsize=13)
    axes[0].set_xlabel('Grade')
    axes[0].set_ylabel('Number of patients')
    for i, v in enumerate(grade_counts.values):
        axes[0].text(i, v + 3, str(v), ha='center', fontsize=10)

# Binary distribution
if 'derived_clavien_bin' in df.columns:
    bin_counts = df['derived_clavien_bin'].value_counts(dropna=False).sort_index()
    colors = ['#88BBDD', '#E07050']
    labels = ['Minor (1-2)', 'Major (3-7)', 'Missing']
    axes[1].pie(bin_counts.values, labels=[labels[int(i)] if not pd.isna(i) else 'Missing'
                for i in bin_counts.index],
                autopct='%1.1f%%', colors=colors, startangle=90)
    axes[1].set_title('Minor vs Major Complications', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()


---
## 3. Missing Data Overview

Before diving into individual variables, we look at the overall missing data picture across all predictors.

**Decision rule used in this study:**
- Variables with **>50% missing** are excluded from the main model
- Variables with **20-50% missing** are included but flagged -- median/mode imputation will be applied
- Variables with **<20% missing** are considered reliable


In [ ]:
# Define all candidate predictors
PREDICTORS = {
    'Demographics': ['age_at_op', 'geslacht'],
    'Fitness & Lung Function': ['asascore', 'fev1vc_percentage'],
    'Nutrition': ['gewverl', 'lengte', 'gewicht', 'BMI', 'dietist'],
    'Comorbidities': ['comhartfaal', 'comaf', 'comcarritme', 'comlong', 'comperif',
                      'comcva', 'comdem', 'comparalys', 'combind', 'comgiulcus',
                      'comlever', 'compancr', 'comdiam', 'comnier', 'comhiv',
                      'commalig', 'commyo', 'roken', 'packyears'],
    'Prior Surgery': ['anamok', 'anamok1', 'anamok2', 'anamok3',
                      'anamok4', 'anamok5', 'anamok6', 'anamok7'],
    'Tumor & Disease': ['tumlok1', 'tumhist', 'tumkeus', 'scorect', 'scorecn', 'scorecm'],
    'Neoadjuvant': ['neoadj', 'neoadjtype', 'neoadjsch', 'neoadjaf'],
    'Surgical': ['typok', 'aardok', 'procok', 'conversie', 'duur_operatie',
                 'bloedverlies', 'resadd', 'curaok', 'resecok', 'salvok',
                 'reslmm', 'recontype', 'analig', 'emr']
}

# Flatten to single list of present columns
all_predictors = [col for group in PREDICTORS.values() for col in group if col in df.columns]

# Missing data summary
missing = pd.DataFrame({
    'Variable': all_predictors,
    'N missing': [df[c].isna().sum() for c in all_predictors],
    'N present': [df[c].notna().sum() for c in all_predictors],
    '% missing': [df[c].isna().mean() * 100 for c in all_predictors]
}).sort_values('% missing', ascending=False)

# Flag status
def flag(pct):
    if pct > 50: return '  Exclude'
    elif pct > 20: return '  Impute (flag)'
    else: return '  OK'

missing['Status'] = missing['% missing'].apply(flag)

print(missing[missing['% missing'] > 0].to_string(index=False))
print(f"\nVariables with 0% missing: {(missing['% missing'] == 0).sum()}")


In [ ]:
# Visual: missing data bar chart
fig, ax = plt.subplots(figsize=(14, 8))

miss_plot = missing[missing['% missing'] > 0].sort_values('% missing', ascending=True)
colors = ['#E07050' if p > 50 else '#F0A860' if p > 20 else '#88BBDD'
          for p in miss_plot['% missing']]

bars = ax.barh(miss_plot['Variable'], miss_plot['% missing'], color=colors, edgecolor='white')
ax.axvline(50, color='red', linestyle='--', linewidth=1.5, label='50% threshold (exclude)')
ax.axvline(20, color='orange', linestyle='--', linewidth=1.5, label='20% threshold (flag)')
ax.set_xlabel('% Missing', fontsize=12)
ax.set_title('Missing Data by Predictor Variable', fontweight='bold', fontsize=14)
ax.legend(loc='lower right')

for bar, pct in zip(bars, miss_plot['% missing']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()


---
## 4. Continuous Variables

For each continuous predictor we report:
- **N** (non-missing count), **Mean**, **Median**, **Std**, **Min**, **Max**
- **% missing**
- **Distribution histogram** with a red line at the median
- **Outlier check** using the IQR method (values beyond Q1 - 1.5?IQR or Q3 + 1.5?IQR)

Outliers do not necessarily mean exclusion -- they may be clinically valid extreme values -- but they should be inspected before modeling.


In [ ]:
CONTINUOUS = ['age_at_op', 'asascore', 'fev1vc_percentage',
              'gewverl', 'lengte', 'gewicht', 'BMI', 'packyears',
              'duur_operatie', 'bloedverlies', 'neoadjaf']

cont_present = [c for c in CONTINUOUS if c in df.columns]

# Summary table
rows = []
for col in cont_present:
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((s < Q1 - 1.5*IQR) | (s > Q3 + 1.5*IQR)).sum()
    rows.append({
        'Variable': col,
        'N': len(s),
        '% missing': round(df[col].isna().mean() * 100, 1),
        'Mean': round(s.mean(), 2),
        'Median': round(s.median(), 2),
        'Std': round(s.std(), 2),
        'Min': round(s.min(), 2),
        'Max': round(s.max(), 2),
        'N outliers (IQR)': n_out
    })

cont_summary = pd.DataFrame(rows)
print(cont_summary.to_string(index=False))


In [ ]:
# Distribution plots for continuous variables
n_cols = 3
n_rows = int(np.ceil(len(cont_present) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cont_present):
    ax = axes[i]
    data = pd.to_numeric(df[col], errors='coerce').dropna()
    ax.hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.5,
               label=f'Median: {data.median():.1f}')
    ax.set_title(col, fontweight='bold', fontsize=12)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    miss_pct = df[col].isna().mean() * 100
    ax.text(0.98, 0.95, f'Missing: {miss_pct:.1f}%',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=9, color='gray')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Continuous Predictors', fontweight='bold', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Boxplots to visualise outliers
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cont_present):
    ax = axes[i]
    data = pd.to_numeric(df[col], errors='coerce').dropna()
    ax.boxplot(data, vert=True, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(col, fontweight='bold', fontsize=12)
    ax.set_ylabel('Value')

    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)).sum()
    ax.text(0.98, 0.95, f'Outliers: {n_out}',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=9, color='darkorange' if n_out > 0 else 'gray')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplots of Continuous Predictors (outliers visible as dots)', fontweight='bold', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()


---
## 5. Binary & Categorical Variables

For binary flags (comorbidities, prior surgery, etc.) we report:
- **N = 1** (patients where the condition is present)
- **% = 1** (prevalence)
- **% missing**

For categorical variables (tumor location, histology, staging, etc.) we report the full value distribution.

A binary variable with **very low prevalence (<2%)** may not be informative enough for modeling -- the model will rarely see positive cases during training.


In [ ]:
# Binary variables
BINARY = ['geslacht', 'dietist', 'roken',
          'comhartfaal', 'comaf', 'comcarritme', 'comlong', 'comperif',
          'comcva', 'comdem', 'comparalys', 'combind', 'comgiulcus',
          'comlever', 'compancr', 'comdiam', 'comnier', 'comhiv',
          'commalig', 'commyo',
          'anamok', 'anamok1', 'anamok2', 'anamok3',
          'anamok4', 'anamok5', 'anamok6', 'anamok7',
          'neoadj', 'conversie', 'resadd', 'curaok', 'salvok', 'emr']

bin_present = [c for c in BINARY if c in df.columns]

rows = []
for col in bin_present:
    n_one = (df[col] == 1).sum()
    pct_one = n_one / df[col].notna().sum() * 100 if df[col].notna().sum() > 0 else 0
    miss = df[col].isna().mean() * 100
    flag = '  Low prevalence (<2%)' if pct_one < 2 else ('  Missing >20%' if miss > 20 else ' ')
    rows.append({'Variable': col, 'N (=1)': int(n_one),
                 '% positive': round(pct_one, 1),
                 '% missing': round(miss, 1),
                 'Status': flag})

bin_summary = pd.DataFrame(rows)
print(bin_summary.to_string(index=False))


In [ ]:
# Visual: prevalence of binary comorbidity variables
comorbidities = ['comhartfaal', 'comaf', 'comcarritme', 'comlong', 'comperif',
                 'comcva', 'comdem', 'comparalys', 'combind', 'comgiulcus',
                 'comlever', 'compancr', 'comdiam', 'comnier', 'comhiv',
                 'commalig', 'commyo']
com_present = [c for c in comorbidities if c in df.columns]

labels = {
    'comhartfaal': 'Heart failure', 'comaf': 'Atrial fibrillation',
    'comcarritme': 'Other arrhythmia', 'comlong': 'Lung disease (COPD)',
    'comperif': 'Peripheral vascular', 'comcva': 'Stroke',
    'comdem': 'Dementia', 'comparalys': 'Paralysis',
    'combind': 'Connective tissue', 'comgiulcus': 'GI ulcer',
    'comlever': 'Liver disease', 'compancr': 'Pancreatic disease',
    'comdiam': 'Diabetes', 'comnier': 'Renal disease',
    'comhiv': 'HIV', 'commalig': 'Prior malignancy', 'commyo': 'Myopathy'
}

prevalences = [(labels.get(c, c), (df[c] == 1).sum() / df[c].notna().sum() * 100)
               for c in com_present]
prevalences.sort(key=lambda x: x[1], reverse=True)
names, vals = zip(*prevalences)

fig, ax = plt.subplots(figsize=(13, 7))
colors = ['#E07050' if v < 2 else '#88BBDD' for v in vals]
bars = ax.barh(names, vals, color=colors, edgecolor='white')
ax.axvline(2, color='red', linestyle='--', linewidth=1, alpha=0.6, label='2% threshold')
ax.set_xlabel('Prevalence (%)', fontsize=12)
ax.set_title('Comorbidity Prevalence', fontweight='bold', fontsize=14)
ax.legend()
for bar, val in zip(bars, vals):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Categorical variables -- value distributions
CATEGORICAL = ['tumlok1', 'tumhist', 'tumkeus', 'scorect', 'scorecn', 'scorecm',
               'neoadjtype', 'neoadjsch', 'typok', 'aardok', 'procok',
               'resecok', 'reslmm', 'recontype', 'analig']

cat_present = [c for c in CATEGORICAL if c in df.columns]

n_cols = 3
n_rows = int(np.ceil(len(cat_present) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cat_present):
    ax = axes[i]
    vc = df[col].value_counts(dropna=False).head(10)
    colors_bar = ['#C0C0C0' if pd.isna(k) else 'steelblue' for k in vc.index]
    ax.bar([str(k) for k in vc.index], vc.values, color=colors_bar, edgecolor='white')
    ax.set_title(col, fontweight='bold', fontsize=12)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
    miss = df[col].isna().mean() * 100
    ax.text(0.98, 0.95, f'Missing: {miss:.1f}%',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=9, color='gray')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Categorical Predictors', fontweight='bold', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()


---
## 6. Univariate Association with Complication Outcome

Here we check whether each predictor shows a meaningful association with the main outcome (`compl` = any complication).

- **Binary/categorical variables**: compare complication rates between groups (chi-square test)

This is purely exploratory -- it does **not** define which variables to keep or drop.
A non-significant result here does not mean the variable is useless (it may interact with others in the full model).

In [ ]:
if 'compl' in df.columns:
    # Binary vs compl
    print("\nBINARY VARIABLES vs compl (complication rate by group)")
    print("=" * 65)
    print(f"{'Variable':<22} {'Rate if 0':>12} {'Rate if 1':>12} {'p-value':>10}")
    print("=" * 65)

    for col in bin_present:
        if col == 'geslacht':
            continue
        grp0 = df.loc[df[col] == 0, 'compl'].dropna()
        grp1 = df.loc[df[col] == 1, 'compl'].dropna()
        if len(grp0) > 5 and len(grp1) > 5:
            ct = pd.crosstab(df[col], df['compl'])
            if ct.shape == (2, 2):
                chi2, p, _, _ = stats.chi2_contingency(ct)
                sig = ' *' if p < 0.05 else ''
                print(f"{col:<22} {grp0.mean()*100:>11.1f}% {grp1.mean()*100:>11.1f}% {p:>10.4f}{sig}")
    print("\n* p < 0.05")


---
## 7. Correlation Between Continuous Predictors

Highly correlated predictors (r > 0.8) may cause **multicollinearity** in logistic regression models.
This is less of a concern for tree-based models like XGBoost, but good to flag either way.

If two variables are highly correlated, consider keeping only the more clinically interpretable one.


In [ ]:
# ── Focused correlation heatmap — key clinical variables ─────────────────────
focus_map = {
    'age_at_op':         'Age at surgery',
    'BMI':               'BMI',
    'asascore':          'ASA score',
    'fev1vc_percentage': 'FEV1/VC %',
    'gewverl':           'Preop. weight loss',
    'duur_operatie':     'Op. duration',
    'bloedverlies':      'Blood loss',
}

# CCI: compute from components if available
cci_components = [
    'comhartfaal','comaf','comcarritme','comlong','comperif','comcva',
    'comdem','comparalys','combind','comgiulcus','comlever','compancr',
    'comdiam','comnier','comhiv','commalig',
]
cci_present = [c for c in cci_components if c in df.columns]
if cci_present:
    df['_cci_count'] = (
        df[cci_present].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
    )
    focus_map['_cci_count'] = 'Comorbidity count'

focus_present = {k: v for k, v in focus_map.items() if k in df.columns}

corr_df = df[list(focus_present.keys())].apply(pd.to_numeric, errors='coerce')
corr_df.columns = list(focus_present.values())
corr_mat = corr_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax,
            annot_kws={'size': 10}, linewidths=0.5)
ax.set_title('Correlation — Key Clinical Variables', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# Flag noteworthy pairs
print("\nCorrelation pairs |r| > 0.40:")
print("-" * 50)
found = False
for i in range(len(corr_mat.columns)):
    for j in range(i):
        r = corr_mat.iloc[i, j]
        if abs(r) > 0.40:
            print(f"  {corr_mat.columns[i]:25} <-> {corr_mat.columns[j]:25}  r = {r:.2f}")
            found = True
if not found:
    print("  None above threshold.")


---
## 8. Predictor Readiness Summary

This is the final checklist. For each predictor we assess:

| Status | Meaning |
|---|---|
|   Ready | < 20% missing, no major issues |
|   Needs attention | 20-50% missing, or low prevalence, or outliers -- can still be used with care |
|   Exclude | > 50% missing or fundamental data problem |

Variables marked   will be handled by imputation in the preprocessing pipeline.


In [ ]:
rows = []

for group, cols in PREDICTORS.items():
    for col in cols:
        if col not in df.columns:
            rows.append({'Group': group, 'Variable': col,
                        'Type': 'N/A', 'N present': 0,
                        '% missing': 100.0, 'Notes': 'Column not found in dataset',
                        'Status': '  Exclude'})
            continue

        miss_pct = df[col].isna().mean() * 100
        n_present = df[col].notna().sum()
        dtype = 'continuous' if pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() > 10 else 'binary/categorical'

        notes = []
        status = '  Ready'

        if miss_pct > 50:
            status = '  Exclude'
            notes.append(f'{miss_pct:.0f}% missing')
        elif miss_pct > 20:
            status = '  Needs attention'
            notes.append(f'{miss_pct:.0f}% missing -- impute')

        if dtype == 'binary/categorical':
            pct_one = (df[col] == 1).sum() / n_present * 100 if n_present > 0 else 0
            if pct_one < 2 and miss_pct <= 50:
                if status == '  Ready':
                    status = '  Needs attention'
                notes.append(f'Low prevalence ({pct_one:.1f}%)')
        else:
            s = df[col].dropna()
            if len(s) > 0:
                Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
                IQR = Q3 - Q1
                n_out = ((s < Q1 - 1.5*IQR) | (s > Q3 + 1.5*IQR)).sum()
                if n_out > len(s) * 0.05:
                    if status == '  Ready':
                        status = '  Needs attention'
                    notes.append(f'{n_out} outliers ({n_out/len(s)*100:.1f}%)')

        rows.append({
            'Group': group,
            'Variable': col,
            'Type': dtype,
            'N present': int(n_present),
            '% missing': round(miss_pct, 1),
            'Notes': '; '.join(notes) if notes else '--',
            'Status': status
        })

readiness = pd.DataFrame(rows)

# Print by group
for group in PREDICTORS.keys():
    sub = readiness[readiness['Group'] == group]
    print(f"\n{'='*70}")
    print(f"  {group.upper()}")
    print(f"{'='*70}")
    print(sub[['Variable', 'Type', 'N present', '% missing', 'Notes', 'Status']].to_string(index=False))


In [ ]:
# Final counts
print("\n" + "="*40)
print("READINESS SUMMARY")
print("="*40)
vc = readiness['Status'].value_counts()
for status, count in vc.items():
    print(f"  {status}: {count} variables")

print(f"\nTotal candidate predictors: {len(readiness)}")
print(f"Ready for modeling: {(readiness['Status'] == '  Ready').sum()}")
print(f"Usable with attention: {(readiness['Status'] == '  Needs attention').sum()}")
print(f"Excluded: {(readiness['Status'] == '  Exclude').sum()}")


In [ ]:
# Visual readiness summary
fig, ax = plt.subplots(figsize=(14, max(6, len(readiness) * 0.35)))

color_map = {'  Ready': '#4CAF50', '  Needs attention': '#FF9800', '  Exclude': '#F44336'}
y_pos = range(len(readiness))

for i, (_, row) in enumerate(readiness.iterrows()):
    color = color_map.get(row['Status'], 'gray')
    ax.barh(i, row['N present'], color=color, alpha=0.75, edgecolor='white')
    ax.text(row['N present'] + 5, i,
            f"{row['Variable']} -- {row['Notes'] if row['Notes'] != '--' else 'OK'}",
            va='center', fontsize=8.5)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(readiness['Variable'], fontsize=9)
ax.set_xlabel('N patients with data', fontsize=11)
ax.set_title('Predictor Readiness Overview', fontweight='bold', fontsize=14)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in color_map.items()]
ax.legend(handles=legend_elements, loc='lower right')
ax.axvline(df.shape[0] * 0.5, color='red', linestyle='--', linewidth=1, alpha=0.5, label='50% threshold')

plt.tight_layout()
plt.show()


---
## 9. Conclusions & Recommended Actions

Based on the analysis above, here is what to do before modeling:

### Variables that need preprocessing action

| Action | Variables |
|---|---|
| **Compute** | `age_at_op` (from gebdat + datok), `BMI` (from gewicht + lengte) |
| **Median imputation** | All continuous variables with missing values |
| **Mode imputation** | All binary/categorical variables with missing values |
| **Treat as ordinal** | `scorect`, `scorecn`, `scorecm` (T1 < T2 < T3 < T4, etc.) |
| **One-hot encode** | `tumlok1`, `tumhist`, `tumkeus`, `neoadjtype`, `typok`, `aardok`, `procok`, `recontype`, `analig` |
| **Log-transform** | `duur_operatie`, `bloedverlies` if heavily right-skewed |
| **Check low prevalence** | Binary variables with <2% positive rate -- consider grouping or excluding |

### Sensitivity analysis (recommended)
Run the model twice:
1. **Main model**: full 1,181 patients without `fev1vc_percentage` (drop missing)
2. **Sensitivity model**: 599-patient subset with all lung function variables included

If AUC improves meaningfully in the sensitivity model, this is evidence that complete spirometry data would strengthen the model.

### Next step
Once preprocessing is complete, proceed to:
1. Penalised logistic regression (LASSO) for feature selection
2. XGBoost with SHAP values for feature importance
3. Stratified 5-fold cross-validation
4. Report AUC-ROC, balanced accuracy, and calibration plots